In [ ]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 1.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
# import category_encoders as ce



In [3]:
# Исходный датасет
df = pd.DataFrame({
    'район': ['Центр', 'Север', 'Юг', 'Север', 'Центр'],
    'тип_дома': ['Кирпич', 'Панель', 'Монолит', 'Кирпич', 'Панель'],
    'состояние': ['Отличное', 'Требует ремонта', 'Хорошее', 'Удовлетворительное', 'Хорошее'],
    'цена_млн': [12.5, 7.2, 9.8, 8.1, 10.3]
})

print("Исходный датасет:")
print(df)
print()

Исходный датасет:
   район тип_дома           состояние  цена_млн
0  Центр   Кирпич            Отличное      12.5
1  Север   Панель     Требует ремонта       7.2
2     Юг  Монолит             Хорошее       9.8
3  Север   Кирпич  Удовлетворительное       8.1
4  Центр   Панель             Хорошее      10.3



Label Encoding

In [ ]:
df_label = df.copy()

le = LabelEncoder()
df_label['район'] = le.fit_transform(df_label['район'])
df_label['тип_дома'] = le.fit_transform(df_label['тип_дома'])


print("1. Label Encoding:")
print(df_label)
print()

1. Label Encoding:
   район  тип_дома  цена_млн
0      1         0      12.5
1      0         2       7.2
2      2         1       9.8
3      0         0       8.1
4      1         2      10.3



One-Hot Encoding (полный)

In [ ]:
df_ohe = df.copy()

df_ohe = pd.get_dummies(df_ohe, columns=['район'])

print("2. One-Hot Encoding (полный):")
print(df_ohe)
print()

2. One-Hot Encoding (полный):
  тип_дома  цена_млн  район_Север  район_Центр  район_Юг
0   Кирпич      12.5        False         True     False
1   Панель       7.2         True        False     False
2  Монолит       9.8        False        False      True
3   Кирпич       8.1         True        False     False
4   Панель      10.3        False         True     False



In [ ]:
# Вариант через sklearn
encoder_ohe_full = OneHotEncoder(sparse_output=False)
encoded_full = encoder_ohe_full.fit_transform(df[['район']])
encoded_df_full = pd.DataFrame(
    encoded_full,
    columns=encoder_ohe_full.get_feature_names_out(['район'])
)
df_ohe_sklearn_full = pd.concat(
    [df[['тип_дома','цена_млн']].reset_index(drop=True), encoded_df_full],
    axis=1
)
print("2a. One-Hot Encoding полный через sklearn:")
print(df_ohe_sklearn_full)
print()

2a. One-Hot Encoding полный через sklearn:
  тип_дома  цена_млн  район_Север  район_Центр  район_Юг
0   Кирпич      12.5          0.0          1.0       0.0
1   Панель       7.2          1.0          0.0       0.0
2  Монолит       9.8          0.0          0.0       1.0
3   Кирпич       8.1          1.0          0.0       0.0
4   Панель      10.3          0.0          1.0       0.0



One-Hot Encoding (drop_first)

In [ ]:
df_ohe_drop = df.copy()

df_ohe_drop = pd.get_dummies(df_ohe_drop, columns=['район'], drop_first=True)

print("3. One-Hot Encoding (drop_first=True):")
print(df_ohe_drop)
print()

# Вариант через sklearn
encoder_ohe = OneHotEncoder(drop='first', sparse_output=False)
encoded = encoder_ohe.fit_transform(df[['район']])
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder_ohe.get_feature_names_out(['район'])
)
df_ohe_sklearn = pd.concat(
    [df[['тип_дома', 'цена_млн']].reset_index(drop=True), encoded_df],
    axis=1
)
print("3a. One-Hot Encoding через sklearn:")
print(df_ohe_sklearn)
print()

3. One-Hot Encoding (drop_first=True):
  тип_дома  цена_млн  район_Центр  район_Юг
0   Кирпич      12.5         True     False
1   Панель       7.2        False     False
2  Монолит       9.8        False      True
3   Кирпич       8.1        False     False
4   Панель      10.3         True     False

3a. One-Hot Encoding через sklearn:
  тип_дома  цена_млн  район_Центр  район_Юг
0   Кирпич      12.5          1.0       0.0
1   Панель       7.2          0.0       0.0
2  Монолит       9.8          0.0       1.0
3   Кирпич       8.1          0.0       0.0
4   Панель      10.3          1.0       0.0



Binary Encoding

In [ ]:
df_binary = df.copy()

encoder_bin = ce.BinaryEncoder(cols=['район', 'тип_дома'])
df_binary = encoder_bin.fit_transform(df_binary)

print("4. Binary Encoding:")
print(df_binary)
print()

4. Binary Encoding:
   район_0  район_1  тип_дома_0  тип_дома_1  цена_млн
0        0        1           0           1      12.5
1        1        0           1           0       7.2
2        1        1           1           1       9.8
3        1        0           0           1       8.1
4        0        1           1           0      10.3



Ordinal Encoding

In [ ]:
df_ordinal = df.copy()

encoder_ord = OrdinalEncoder(
    categories=[['Требует ремонта', 'Удовлетворительное', 'Хорошее', 'Отличное']]
)
df_ordinal[['состояние']] = encoder_ord.fit_transform(df_ordinal[['состояние']])

print("5. Ordinal Encoding:")
print(df_ordinal)

5. Ordinal Encoding:
   район тип_дома  состояние  цена_млн
0  Центр   Кирпич        3.0      12.5
1  Север   Панель        0.0       7.2
2     Юг  Монолит        2.0       9.8
3  Север   Кирпич        1.0       8.1
4  Центр   Панель        2.0      10.3


Target (Mean) Encoder

In [7]:
from sklearn.preprocessing import TargetEncoder

encoder_target = TargetEncoder(cv=3, smooth=0.2)
df_target = df.copy()
df_target['район'] = encoder_target.fit_transform(
    df_target[['район']], df_target['цена_млн']
)

print("6. Target Encoding:")
print(df_target)

6. Target Encoding:
       район тип_дома           состояние  цена_млн
0  10.150000   Кирпич            Отличное      12.5
1   8.316667   Панель     Требует ремонта       7.2
2   9.525000  Монолит             Хорошее       9.8
3   7.638889   Кирпич  Удовлетворительное       8.1
4  12.055556   Панель             Хорошее      10.3


LOO

In [ ]:
import category_encoders as ce

encoder_loo = ce.LeaveOneOutEncoder(cols=['район', 'тип_дома'])
df_loo = df.copy()

# Важно: fit_transform требует целевую переменную
df_loo[['район', 'тип_дома']] = encoder_loo.fit_transform(
    df_loo[['район', 'тип_дома']], df_loo['цена_млн']
)

print("7. LOO Encoding:")
print(df_loo)

7. LOO Encoding:
   район  тип_дома           состояние  цена_млн
0  10.30      8.10            Отличное      12.5
1   8.10     10.30     Требует ремонта       7.2
2   9.58      9.58             Хорошее       9.8
3   7.20     12.50  Удовлетворительное       8.1
4  12.50      7.20             Хорошее      10.3


Масштабирование признаков

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

df_scaled_std = df.copy()

# StandardScaler
scaler_std = StandardScaler()
df_scaled_std[['цена_млн']] = scaler_std.fit_transform(df_scaled_std[['цена_млн']])



,район,тип_дома,состояние,цена_млн
0,Центр,Кирпич,Отличное,1.586020
1,Север,Панель,Требует ремонта,-1.292715
2,Юг,Монолит,Хорошее,0.119495
3,Север,Кирпич,Удовлетворительное,-0.803873
4,Центр,Панель,Хорошее,0.391073


In [ ]:
print(df_scaled_std)

   район тип_дома           состояние  цена_млн
0  Центр   Кирпич            Отличное  1.586020
1  Север   Панель     Требует ремонта -1.292715
2     Юг  Монолит             Хорошее  0.119495
3  Север   Кирпич  Удовлетворительное -0.803873
4  Центр   Панель             Хорошее  0.391073


In [ ]:
df_scaled_minmax = df.copy()

# MinMaxScaler
scaler_mm = MinMaxScaler()
df_scaled_minmax[['цена_млн']] = scaler_mm.fit_transform(df_scaled_minmax[['цена_млн']])

In [ ]:
print(df_scaled_minmax)

   район тип_дома           состояние  цена_млн
0  Центр   Кирпич            Отличное  1.000000
1  Север   Панель     Требует ремонта  0.000000
2     Юг  Монолит             Хорошее  0.490566
3  Север   Кирпич  Удовлетворительное  0.169811
4  Центр   Панель             Хорошее  0.584906


In [ ]:
from sklearn.preprocessing import RobustScaler

df_scaled_robust = df.copy()

scaler_rob = RobustScaler()
df_scaled_robust[['цена_млн']] = scaler_rob.fit_transform(df_scaled_robust[['цена_млн']])

In [ ]:
print(df_scaled_robust)

   район тип_дома           состояние  цена_млн
0  Центр   Кирпич            Отличное  1.227273
1  Север   Панель     Требует ремонта -1.181818
2     Юг  Монолит             Хорошее  0.000000
3  Север   Кирпич  Удовлетворительное -0.772727
4  Центр   Панель             Хорошее  0.227273


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test = train_test_split(df, test_size=0.2, random_state=42)

scaler = StandardScaler()

# fit только на трейне, transform — на обоих
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # не fit_transform!

Логарифмирование

In [ ]:
import numpy as np

df['цена_log+1'] = np.log1p(df['цена_млн'])
df['цена_log'] = np.log(df['цена_млн'])

In [ ]:
print(df)

   район тип_дома           состояние  цена_млн  цена_log  цена_log+1
0  Центр   Кирпич            Отличное      12.5  2.525729    2.602690
1  Север   Панель     Требует ремонта       7.2  1.974081    2.104134
2     Юг  Монолит             Хорошее       9.8  2.282382    2.379546
3  Север   Кирпич  Удовлетворительное       8.1  2.091864    2.208274
4  Центр   Панель             Хорошее      10.3  2.332144    2.424803


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(df[['площадь']])

Временные признаки

In [ ]:
import pandas as pd
import numpy as np

df_dates = pd.DataFrame({
    'дата_заявки': pd.to_datetime([
        '2024-01-15', '2024-03-22', '2024-07-04',
        '2024-11-30', '2024-12-31', '2024-06-01'
    ]),
    'дата_заезда': pd.to_datetime([
        '2024-02-01', '2024-04-01', '2024-07-10',
        '2024-12-15', '2025-01-05', '2024-06-15'
    ]),
    'район': ['Центр', 'Север', 'Юг', 'Центр', 'Север', 'Юг'],
    'цена_аренда': [45000, 28000, 32000, 50000, 30000, 35000]
})

print(df_dates)

  дата_заявки дата_заезда  район  цена_аренда
0  2024-01-15  2024-02-01  Центр        45000
1  2024-03-22  2024-04-01  Север        28000
2  2024-07-04  2024-07-10     Юг        32000
3  2024-11-30  2024-12-15  Центр        50000
4  2024-12-31  2025-01-05  Север        30000
5  2024-06-01  2024-06-15     Юг        35000


In [ ]:
df_dates['год']          = df_dates['дата_заявки'].dt.year
df_dates['месяц']        = df_dates['дата_заявки'].dt.month
df_dates['день']         = df_dates['дата_заявки'].dt.day
df_dates['день_недели']  = df_dates['дата_заявки'].dt.dayofweek  # 0 = понедельник
df_dates['квартал']      = df_dates['дата_заявки'].dt.quarter
df_dates['номер_недели'] = df_dates['дата_заявки'].dt.isocalendar().week

print(df_dates[['дата_заявки', 'год', 'месяц', 'день', 'день_недели', 'квартал']])

  дата_заявки   год  месяц  день  день_недели  квартал
0  2024-01-15  2024      1    15            0        1
1  2024-03-22  2024      3    22            4        1
2  2024-07-04  2024      7     4            3        3
3  2024-11-30  2024     11    30            5        4
4  2024-12-31  2024     12    31            1        4
5  2024-06-01  2024      6     1            5        2


In [ ]:
# Выходной день
df_dates['выходной'] = df_dates['день_недели'].isin([5, 6]).astype(int)

# Праздник
import holidays
ru_holidays = holidays.Russia()
df_dates['праздник'] = df_dates['дата_заявки'].apply(
    lambda x: int(x in ru_holidays)
)


# Сезон
df_dates['сезон'] = df_dates['месяц'].map({
    12: 'зима', 1: 'зима',  2: 'зима',
     3: 'весна', 4: 'весна', 5: 'весна',
     6: 'лето',  7: 'лето',  8: 'лето',
     9: 'осень', 10: 'осень', 11: 'осень'
})

print(df_dates[['дата_заявки', 'выходной', 'праздник', 'сезон']])

  дата_заявки  выходной  праздник  сезон
0  2024-01-15         0         0   зима
1  2024-03-22         0         0  весна
2  2024-07-04         0         0   лето
3  2024-11-30         1         0  осень
4  2024-12-31         0         1   зима
5  2024-06-01         1         0   лето


In [ ]:
# Сколько дней между заявкой и заездом
df_dates['дней_до_заезда'] = (
    df_dates['дата_заезда'] - df_dates['дата_заявки']
).dt.days

print(df_dates[['дата_заявки', 'дата_заезда', 'дней_до_заезда']])

  дата_заявки дата_заезда  дней_до_заезда
0  2024-01-15  2024-02-01              17
1  2024-03-22  2024-04-01              10
2  2024-07-04  2024-07-10               6
3  2024-11-30  2024-12-15              15
4  2024-12-31  2025-01-05               5
5  2024-06-01  2024-06-15              14
